# Аналіз ефективності реклами в соціальних мережах
## Покращена версія — 5 ключових оптимізацій

### Що змінено:
1. **Оптимізація scoring**: замість accuracy → F1-macro + per-class F1 з confusion matrix
2. **Балансування датасету**: Impression = Passive Views через downsampling
3. **Рівномірний розподіл активних взаємодій**: SMOTE-подібний oversampling всередині Stage 2
4. **Порівняння кількох моделей на кожному етапі**: LR, RF, ExtraTrees, GradientBoosting
5. **Автоматичне налаштування гіперпараметрів**: RandomizedSearchCV з широкими сітками

---

## 1. Встановлення залежностей та імпорт бібліотек

In [1]:
!pip install kagglehub duckdb xgboost matplotlib seaborn -q

In [2]:
import kagglehub
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from os import path
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, cross_validate,
    StratifiedKFold, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, f1_score, make_scorer
)
from scipy.stats import randint, uniform

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️  XGBoost не встановлено — буде використано GradientBoosting як замінник')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.titlesize': 15, 'axes.labelsize': 12})
RANDOM_STATE = 42
print('✅ Всі бібліотеки успішно імпортовано')

✅ Всі бібліотеки успішно імпортовано


---
## 2. Завантаження та підготовка даних

In [3]:
print('Завантаження датасету...')
kaggle_path = kagglehub.dataset_download('alperenmyung/social-media-advertisement-performance')
db_path = path.join(kaggle_path, 'ad_campaign_db.sqlite')

con = duckdb.connect()
con.execute('INSTALL sqlite; LOAD sqlite;')
con.execute(f"ATTACH '{db_path}' AS sqlite_db (TYPE SQLITE);")

tables = con.execute("SHOW TABLES FROM sqlite_db").df()
print('\nТаблиці в базі даних:')
print(tables)

Завантаження датасету...

Таблиці в базі даних:
        name
0  ad_events
1        ads
2  campaigns
3      users


In [4]:
query = """
SELECT
    u.user_id, u.user_gender, u.user_age, u.age_group, u.country,
    u.interests AS user_interests,
    a.ad_platform, a.ad_type, a.target_gender, a.target_age_group,
    a.target_interests AS ad_target_interests,
    c.total_budget, c.duration_days,
    e.timestamp, e.day_of_week, e.time_of_day, e.event_type
FROM sqlite_db.ad_events e
JOIN sqlite_db.users     u ON e.user_id = u.user_id
JOIN sqlite_db.ads       a ON e.ad_id   = a.ad_id
JOIN sqlite_db.campaigns c ON a.campaign_id = c.campaign_id;
"""

df = con.execute(query).df()
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f'Дані завантажено: {df.shape[0]:,} рядків × {df.shape[1]} стовпців')
print(f'\nРозподіл event_type:')
print(df['event_type'].value_counts())
df.head(3)

Дані завантажено: 403,967 рядків × 17 стовпців

Розподіл event_type:
event_type
Impression    343157
Click          40495
Like           12145
Comment         4142
Purchase        2050
Share           1978
Name: count, dtype: int64


,user_id,user_gender,user_age,age_group,country,user_interests,ad_platform,ad_type,target_gender,target_age_group,ad_target_interests,total_budget,duration_days,timestamp,day_of_week,time_of_day,event_type
0,d03d7,Male,20,18-24,Mexico,"fitness, fashion, lifestyle",Facebook,Carousel,All,25-34,art,71038.28,36,2025-08-04 11:07:30,Monday,Morning,Impression
1,9f579,Male,30,25-34,United States,"news, lifestyle, fashion",Facebook,Video,Female,All,technology,26001.67,84,2025-05-07 15:31:24,Wednesday,Afternoon,Impression
2,f0892,Other,34,25-34,United States,sports,Facebook,Stories,Female,All,finance,45326.60,65,2025-05-16 15:45:16,Friday,Afternoon,Impression


---
## 3. Інженерія ознак

In [5]:
# 1. Відповідність статі
df['is_gender_match'] = (
    (df['user_gender'] == df['target_gender']) |
    (df['target_gender'] == 'All')
).astype(int)

# 2. Відповідність вікової групи
df['is_age_match'] = (
    (df['age_group'] == df['target_age_group']) |
    (df['target_age_group'] == 'All')
).astype(int)

# 3. Кількість спільних інтересів
def get_interest_overlap(row):
    user_int = set(str(row['user_interests']).lower().split(', '))
    ad_int   = set(str(row['ad_target_interests']).lower().split(', '))
    return len(user_int & ad_int)

df['interest_overlap_count'] = df.apply(get_interest_overlap, axis=1)

# 4. Часові ознаки
df['hour']       = df['timestamp'].dt.hour
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# 5. Комбінована релевантність
df['relevance_score'] = (
    df['is_gender_match'] +
    df['is_age_match'] +
    df['interest_overlap_count'].clip(0, 3) / 3
)

print('Нові ознаки створено')
print(df[['is_gender_match','is_age_match','interest_overlap_count','relevance_score']].describe().round(3))

Нові ознаки створено
       is_gender_match  is_age_match  interest_overlap_count  relevance_score
count       403967.000    403967.000              403967.000       403967.000
mean             0.625         0.450                   0.230            1.152
std              0.484         0.498                   0.441            0.706
min              0.000         0.000                   0.000            0.000
25%              0.000         0.000                   0.000            1.000
50%              1.000         0.000                   0.000            1.000
75%              1.000         1.000                   0.000            2.000
max              1.000         1.000                   2.000            2.667


---
## 4. Підготовка матриці ознак (One-Hot Encoding)

In [6]:
cols_to_drop = [
    'user_id', 'timestamp', 'user_interests', 'ad_target_interests',
    'user_gender', 'target_gender', 'age_group', 'target_age_group', 'day_of_week'
]
df_ml = df.drop(columns=cols_to_drop)

categorical_cols = ['ad_platform', 'ad_type', 'time_of_day', 'country']
df_final = pd.get_dummies(df_ml, columns=categorical_cols, drop_first=True)

le = LabelEncoder()
df_final['event_type_enc'] = le.fit_transform(df_final['event_type'])
target_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f'Мапінг цілей: {target_mapping}')

X = df_final.drop(columns=['event_type', 'event_type_enc'])
y = df_final['event_type_enc']
feature_names = X.columns

print(f'Розмір матриці ознак: {X.shape}')

Мапінг цілей: {'Click': np.int64(0), 'Comment': np.int64(1), 'Impression': np.int64(2), 'Like': np.int64(3), 'Purchase': np.int64(4), 'Share': np.int64(5)}
Розмір матриці ознак: (403967, 25)


---
## 5. ВИПРАВЛЕННЯ #2 — Балансування датасету Stage 1

**Проблема:** Impression (~85%) домінує над усіма іншими класами («passive views»).  
**Рішення:** Зберігаємо **всі активні події** + рівну кількість Impression → **50/50 баланс**.

In [7]:
# ─── Бінарна ціль: 0 = Impression, 1 = Engaged ─────────────────────────────
impression_label = list(le.classes_).index('Impression')
y_binary = (y != impression_label).astype(int)

print('Розподіл до балансування:')
vc = y_binary.value_counts(normalize=True)
print(f'  Impression (0): {vc.get(0,0)*100:.1f}%')
print(f'  Engaged    (1): {vc.get(1,0)*100:.1f}%')

# ─── ВИПРАВЛЕННЯ #2: Downsampling Impression до розміру Engaged ─────────────
rng = np.random.RandomState(RANDOM_STATE)

idx_engaged    = np.where(y_binary == 1)[0]
idx_impression = np.where(y_binary == 0)[0]

# Impression і Passive Views — рівна кількість
n_sample = len(idx_engaged)
idx_imp_sample = rng.choice(idx_impression, size=n_sample, replace=False)
idx_s1 = np.concatenate([idx_engaged, idx_imp_sample])
rng.shuffle(idx_s1)

X_s1 = X.iloc[idx_s1].reset_index(drop=True)
y_s1 = y_binary.iloc[idx_s1].reset_index(drop=True)

print(f'\nПісля балансування:')
vc2 = y_s1.value_counts(normalize=True)
print(f'  Impression (0): {vc2.get(0,0)*100:.1f}%')
print(f'  Engaged    (1): {vc2.get(1,0)*100:.1f}%')
print(f'  Розмір датасету Stage 1: {len(X_s1):,} рядків')

# ─── Train/Test split ───────────────────────────────────────────────────────
X_train_s1, X_test_s1, y_train_s1, y_test_s1 = train_test_split(
    X_s1, y_s1, test_size=0.2, stratify=y_s1, random_state=RANDOM_STATE
)
print(f'  Train: {len(X_train_s1):,} | Test: {len(X_test_s1):,}')

Розподіл до балансування:
  Impression (0): 84.9%
  Engaged    (1): 15.1%

Після балансування:
  Impression (0): 50.0%
  Engaged    (1): 50.0%
  Розмір датасету Stage 1: 121,620 рядків
  Train: 97,296 | Test: 24,324


---
## 6. ВИПРАВЛЕННЯ #3 — Рівномірний розподіл активних взаємодій (Stage 2)

**Проблема:** Серед engaged-подій Click (~67%) домінує над Purchase/Share/Comment (<5%).  
**Рішення:** Oversampling міноритарних класів до рівня `min_target_ratio` від Click.  
**Мета:** Модель навчається розрізняти **що впливає на витрати** — не вгадувати розподіл даних.

In [8]:
# ─── Відбираємо тільки engaged події ─────────────────────────────────────────
engaged_mask = (y_binary == 1)
X_engaged = X[engaged_mask.values].reset_index(drop=True)
y_engaged_labels = df.loc[X.index[engaged_mask.values], 'event_type'].values

le_s2 = LabelEncoder()
y_s2_raw = le_s2.fit_transform(y_engaged_labels)

print('Розподіл активних взаємодій ДО балансування:')
for cls, lbl in zip(le_s2.classes_, range(len(le_s2.classes_))):
    cnt = (y_s2_raw == lbl).sum()
    print(f'  {cls:12s}: {cnt:>6,}  ({cnt/len(y_s2_raw)*100:.1f}%)')

# ─── ВИПРАВЛЕННЯ #3: Oversampling міноритарних класів ───────────────────────
# Стратегія: підняти кожен клас до 40% від максимального класу
MIN_RATIO = 0.40  # налаштовуємо: кожен клас буде мати >= 40% від max-класу

class_counts = pd.Series(y_s2_raw).value_counts()
max_count = class_counts.max()
target_count = max(int(max_count * MIN_RATIO), class_counts.min())

balanced_X_list = []
balanced_y_list = []

for cls_idx in range(len(le_s2.classes_)):
    mask_cls = (y_s2_raw == cls_idx)
    X_cls = X_engaged[mask_cls]
    y_cls = y_s2_raw[mask_cls]
    n_cls = len(X_cls)
    
    if n_cls < target_count:
        # Oversample: bootstrap (з поверненням)
        extra_idx = rng.choice(n_cls, size=target_count - n_cls, replace=True)
        X_extra = X_cls.iloc[extra_idx]
        y_extra = y_cls[extra_idx]
        X_cls_bal = pd.concat([X_cls, X_extra], ignore_index=True)
        y_cls_bal = np.concatenate([y_cls, y_extra])
    else:
        # Downsample великі класи щоб не домінували
        # Click залишаємо максимальним, але обмежуємо до 2.5x від target
        cap = min(n_cls, int(target_count * 2.5))
        chosen = rng.choice(n_cls, size=cap, replace=False)
        X_cls_bal = X_cls.iloc[chosen]
        y_cls_bal = y_cls[chosen]
    
    balanced_X_list.append(X_cls_bal)
    balanced_y_list.append(y_cls_bal)

X_s2 = pd.concat(balanced_X_list, ignore_index=True)
y_s2 = np.concatenate(balanced_y_list)

# Перемішуємо
shuffle_idx = rng.permutation(len(X_s2))
X_s2 = X_s2.iloc[shuffle_idx].reset_index(drop=True)
y_s2 = y_s2[shuffle_idx]

print(f'\nРозподіл активних взаємодій ПІСЛЯ балансування:')
for cls, lbl in zip(le_s2.classes_, range(len(le_s2.classes_))):
    cnt = (y_s2 == lbl).sum()
    print(f'  {cls:12s}: {cnt:>6,}  ({cnt/len(y_s2)*100:.1f}%)')
print(f'  Загальний розмір Stage 2: {len(X_s2):,}')

# ─── Train/Test split ───────────────────────────────────────────────────────
# Test береться з ОРИГІНАЛЬНОГО (незбалансованого) розподілу — чесна оцінка!
X_engaged_orig = X_engaged.copy()
y_s2_orig = y_s2_raw.copy()

X_train_s2_orig, X_test_s2, y_train_s2_orig, y_test_s2 = train_test_split(
    X_engaged_orig, y_s2_orig,
    test_size=0.2, stratify=y_s2_orig, random_state=RANDOM_STATE
)

# Train береться зі збалансованого датасету
X_train_s2_bal_part, _, y_train_s2_bal, _ = train_test_split(
    X_s2, y_s2,
    test_size=0.2, stratify=y_s2, random_state=RANDOM_STATE
)
X_train_s2 = X_train_s2_bal_part
y_train_s2 = y_train_s2_bal

print(f'  Train (збалансований): {len(X_train_s2):,} | Test (оригінальний): {len(X_test_s2):,}')

Розподіл активних взаємодій ДО балансування:
  Click       : 40,495  (66.6%)
  Comment     :  4,142  (6.8%)
  Like        : 12,145  (20.0%)
  Purchase    :  2,050  (3.4%)
  Share       :  1,978  (3.3%)

Розподіл активних взаємодій ПІСЛЯ балансування:
  Click       : 40,495  (38.5%)
  Comment     : 16,198  (15.4%)
  Like        : 16,198  (15.4%)
  Purchase    : 16,198  (15.4%)
  Share       : 16,198  (15.4%)
  Загальний розмір Stage 2: 105,287
  Train (збалансований): 84,229 | Test (оригінальний): 12,162


---
## 7. Допоміжні функції: оцінка та візуалізація

### ВИПРАВЛЕННЯ #1 — Scoring оптимізовано по confusion matrix та per-class F1

In [9]:
def evaluate_model(model, X_test, y_test, class_names, stage_name='', fit_needed=False, X_train=None, y_train=None):
    """
    ВИПРАВЛЕННЯ #1: Оцінює модель через confusion matrix та per-class F1.
    Повертає словник метрик для порівняння.
    """
    if fit_needed and X_train is not None:
        model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    acc      = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_per_class = f1_score(y_test, y_pred, average=None, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    # Per-class recall з confusion matrix (diagonal / row sum)
    recall_per_class = np.diag(cm) / cm.sum(axis=1).clip(min=1)
    
    print(f'\n{'='*60}')
    print(f'  {stage_name}')
    print(f'{'='*60}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1-macro  : {f1_macro:.4f}  ← КЛЮЧОВА МЕТРИКА')
    print(f'  F1-weighted: {f1_weighted:.4f}')
    print(f'\n  Per-class F1 (з confusion matrix):')
    for cls, f1, rec in zip(class_names, f1_per_class, recall_per_class):
        print(f'    {cls:12s}: F1={f1:.3f}  Recall={rec:.3f}')
    print()
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))
    
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'f1_per_class': dict(zip(class_names, f1_per_class)),
        'confusion_matrix': cm,
        'y_pred': y_pred
    }


def plot_confusion_matrix(cm, class_names, title, ax, cmap='Blues'):
    """Confusion matrix з абсолютними значеннями та recall по діагоналі."""
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis].clip(min=1)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=class_names, yticklabels=class_names, ax=ax,
                linewidths=0.5)
    # Додаємо recall у дужках на діагоналі
    for i in range(len(class_names)):
        current = ax.texts[i * (len(class_names) + 1)]
        val = cm[i, i]
        rec = cm_norm[i, i]
        current.set_text(f'{val}\n({rec:.0%})')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Прогноз')
    ax.set_ylabel('Реальність')


def compare_models_bar(results_dict, metric='f1_macro', title='', ax=None):
    """Порівняльна гістограма моделей за вказаною метрикою."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    names = list(results_dict.keys())
    vals  = [results_dict[n][metric] for n in names]
    colors = sns.color_palette('viridis', len(names))
    bars = ax.bar(names, vals, color=colors, edgecolor='white')
    ax.set_ylim(0, min(max(vals) * 1.25, 1.0))
    ax.set_title(title or f'{metric} — порівняння моделей', fontweight='bold')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    return ax

print('Допоміжні функції завантажено')

Допоміжні функції завантажено


---
## 8. ВИПРАВЛЕННЯ #4 + #5 — Stage 1: Порівняння моделей з автоматичним налаштуванням

**Binary Classification: Impression (0) vs Engaged (1)**

Кожна модель налаштовується через `RandomizedSearchCV` (ВИПРАВЛЕННЯ #5)  
з оптимізацією по **f1_macro** (ВИПРАВЛЕННЯ #1).

In [10]:
# ─── Imports ────────────────────────────────────────────────────────────────
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

from scipy.stats import randint, loguniform, uniform

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# ─── Config ─────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
SCORING_S1   = 'f1_macro'
CV_FOLDS     = 3   # reduced for speed
N_CANDIDATES = 20  # controls search breadth (instead of N_ITER)

print(f'🔍 Optimized hyperparameter tuning (Stage 1)')
print(f'   Scoring: {SCORING_S1} | CV: {CV_FOLDS}-fold\n')

# ─── CV Strategy ────────────────────────────────────────────────────────────
skf = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# ─── Model Configurations (optimized search spaces) ─────────────────────────
s1_model_configs = {

    'Logistic Regression': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                max_iter=2000,
                class_weight='balanced',
                random_state=RANDOM_STATE,
                n_jobs=1  # prevent oversubscription
            ))
        ]),
        {
            'clf__C': loguniform(1e-3, 10),
            'clf__solver': ['lbfgs'],  # simplified
            'clf__penalty': ['l2'],
        }
    ),

    'Random Forest': (
        RandomForestClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            'n_estimators': randint(100, 300),
            'max_depth': [None, 10, 20],
            'min_samples_leaf': randint(1, 10),
            'max_features': ['sqrt', 0.5],
        }
    ),

    'Extra Trees': (
        ExtraTreesClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            'n_estimators': randint(100, 300),
            'max_depth': [None, 10, 20],
            'min_samples_leaf': randint(1, 10),
            'max_features': ['sqrt', 0.5],
        }
    ),

    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            'n_estimators': randint(100, 250),
            'learning_rate': uniform(0.01, 0.2),
            'max_depth': randint(3, 6),
            'subsample': uniform(0.7, 0.3),
            'min_samples_leaf': randint(10, 40),
        }
    ),
}

# ─── Optional XGBoost ───────────────────────────────────────────────────────
if HAS_XGB:
    s1_model_configs['XGBoost'] = (
        XGBClassifier(
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0,
            tree_method='hist'  # faster
        ),
        {
            'n_estimators': randint(100, 300),
            'learning_rate': uniform(0.01, 0.2),
            'max_depth': randint(3, 6),
            'subsample': uniform(0.7, 0.3),
            'colsample_bytree': uniform(0.6, 0.4),
            'reg_alpha': uniform(0, 0.5),
            'reg_lambda': uniform(1, 3),
        }
    )

# ─── Stage 1: Coarse Search (Halving) ───────────────────────────────────────
s1_best_models = {}
s1_search_results = {}

for name, (estimator, param_dist) in s1_model_configs.items():
    print(f'  🔄 {name}...', end='', flush=True)

    search = HalvingRandomSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_candidates=N_CANDIDATES,
        factor=3,
        resource='n_samples',   # progressively increase data
        max_resources='auto',
        cv=skf,
        scoring=SCORING_S1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    # Fit
    search.fit(X_train_s1, y_train_s1)

    s1_best_models[name] = search.best_estimator_
    s1_search_results[name] = search

    print(f'Best {SCORING_S1}: {search.best_score_:.4f}')

print('\nSearch is compelete')

🔍 Optimized hyperparameter tuning (Stage 1)
   Scoring: f1_macro | CV: 3-fold

  🔄 Logistic Regression...Best f1_macro: 0.4895
  🔄 Random Forest...Best f1_macro: 0.4757
  🔄 Extra Trees...Best f1_macro: 0.4564
  🔄 Gradient Boosting...Best f1_macro: 0.5024
  🔄 XGBoost...Best f1_macro: 0.4972

Search is compelete


In [11]:
# ─── Оцінка всіх моделей Stage 1 на тестовій вибірці ────────────────────────
print('\n📊 STAGE 1 — Порівняння моделей на тестовій вибірці')
s1_test_results = {}
s1_class_names  = ['Impression', 'Engaged']

for name, model in s1_best_models.items():
    res = evaluate_model(
        model, X_test_s1, y_test_s1,
        class_names=s1_class_names,
        stage_name=f'Stage 1 — {name}'
    )
    s1_test_results[name] = res

# Зведена таблиця
s1_summary = pd.DataFrame({
    'Model': list(s1_test_results.keys()),
    'Accuracy':    [r['accuracy']    for r in s1_test_results.values()],
    'F1-macro':    [r['f1_macro']    for r in s1_test_results.values()],
    'F1-weighted': [r['f1_weighted'] for r in s1_test_results.values()],
    'F1-Impression': [r['f1_per_class']['Impression'] for r in s1_test_results.values()],
    'F1-Engaged':    [r['f1_per_class']['Engaged']    for r in s1_test_results.values()],
}).sort_values('F1-macro', ascending=False)

print('\n📋 ЗВЕДЕНА ТАБЛИЦЯ Stage 1:')
print(s1_summary.to_string(index=False, float_format='{:.4f}'.format))

s1_best_name  = s1_summary.iloc[0]['Model']
s1_best_model = s1_best_models[s1_best_name]
print(f'\nНайкраща модель Stage 1: {s1_best_name} (F1-macro={s1_summary.iloc[0]["F1-macro"]:.4f})')


📊 STAGE 1 — Порівняння моделей на тестовій вибірці

  Stage 1 — Logistic Regression
  Accuracy  : 0.5037
  F1-macro  : 0.5036  ← КЛЮЧОВА МЕТРИКА
  F1-weighted: 0.5036

  Per-class F1 (з confusion matrix):
    Impression  : F1=0.513  Recall=0.522
    Engaged     : F1=0.494  Recall=0.485

              precision    recall  f1-score   support

  Impression       0.50      0.52      0.51     12162
     Engaged       0.50      0.49      0.49     12162

    accuracy                           0.50     24324
   macro avg       0.50      0.50      0.50     24324
weighted avg       0.50      0.50      0.50     24324


  Stage 1 — Random Forest
  Accuracy  : 0.4988
  F1-macro  : 0.4988  ← КЛЮЧОВА МЕТРИКА
  F1-weighted: 0.4988

  Per-class F1 (з confusion matrix):
    Impression  : F1=0.496  Recall=0.493
    Engaged     : F1=0.502  Recall=0.505

              precision    recall  f1-score   support

  Impression       0.50      0.49      0.50     12162
     Engaged       0.50      0.50      0.50 

In [ ]:
# ─── Візуалізація Stage 1 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. F1-macro порівняння
compare_models_bar(s1_test_results, 'f1_macro', 'Stage 1: F1-macro по моделях', axes[0])

# 2. Per-class F1
classes_s1 = s1_class_names
model_names = list(s1_test_results.keys())
x = np.arange(len(classes_s1))
width = 0.8 / len(model_names)
for i, (mname, res) in enumerate(s1_test_results.items()):
    vals = [res['f1_per_class'][c] for c in classes_s1]
    axes[1].bar(x + i*width - 0.4 + width/2, vals, width, label=mname)
axes[1].set_xticks(x)
axes[1].set_xticklabels(classes_s1)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Stage 1: Per-class F1 по моделях', fontweight='bold')
axes[1].set_ylabel('F1-score')
axes[1].legend(fontsize=8)

# 3. Confusion matrix найкращої моделі
best_cm_s1 = s1_test_results[s1_best_name]['confusion_matrix']
plot_confusion_matrix(best_cm_s1, s1_class_names,
    f'Confusion Matrix: {s1_best_name}', axes[2], 'Blues')

plt.suptitle(f'Stage 1: Filter Model — Binary Classification (збалансований 50/50)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. ВИПРАВЛЕННЯ #4 + #5 — Stage 2: Action Classifier

**Multiclass: Click / Comment / Like / Purchase / Share**  
Навчання на збалансованих даних (ВИПРАВЛЕННЯ #3), тест на оригінальних.

In [13]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from scipy.stats import uniform, randint

# Optional
if HAS_XGB:
    from xgboost import XGBClassifier

# ─── CONFIG ──────────────────────────────────────────────────────────────────
SCORING_S2   = 'f1_macro'
CV_FOLDS     = 3
RANDOM_STATE = 42

# Halving params
FACTOR = 3

# ─── OPTIONAL: SUBSAMPLING FOR SPEED ─────────────────────────────────────────
X_sub, _, y_sub, _ = train_test_split(
    X_train_s2,
    y_train_s2,
    train_size=0.6,
    stratify=y_train_s2,
    random_state=RANDOM_STATE
)

# ─── MODEL CONFIGS (REDUCED SEARCH SPACE) ─────────────────────────────────────
s2_model_configs = {

    'Logistic Regression': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            ))
        ]),
        {
            'clf__C': uniform(0.01, 5),
            'clf__solver': ['lbfgs'],  # faster
        }
    ),

    'Random Forest': (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1  # avoid nested parallelism
        ),
        {
            'n_estimators': randint(100, 300),
            'max_depth': [None, 20],
            'min_samples_leaf': randint(1, 10),
            'max_features': ['sqrt'],
        }
    ),

    'Extra Trees': (
        ExtraTreesClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            'n_estimators': randint(100, 300),
            'max_depth': [None, 20],
            'min_samples_leaf': randint(1, 10),
            'max_features': ['sqrt'],
        }
    ),
}

# ─── OPTIONAL: XGBOOST (CONTROLLED) ───────────────────────────────────────────
if HAS_XGB:
    s2_model_configs['XGBoost'] = (
        XGBClassifier(
            eval_metric='mlogloss',
            random_state=RANDOM_STATE,
            n_jobs=1,  # IMPORTANT
            verbosity=0,
            num_class=len(le_s2.classes_)
        ),
        {
            'n_estimators': randint(100, 250),
            'learning_rate': uniform(0.03, 0.2),
            'max_depth': randint(3, 6),
            'subsample': uniform(0.6, 0.4),
            'colsample_bytree': uniform(0.6, 0.4),
        }
    )

# ─── SEARCH LOOP ─────────────────────────────────────────────────────────────
print(f'🔍 Fast search for {len(s2_model_configs)} models (Stage 2)')
print(f'   Scoring: {SCORING_S2} | CV: {CV_FOLDS}-fold | Halving factor: {FACTOR}\n')

s2_best_models    = {}
s2_search_results = {}

skf2 = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

for name, (estimator, param_dist) in s2_model_configs.items():
    print(f'  {name}...', end='', flush=True)

    search = HalvingRandomSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        factor=FACTOR,
        cv=skf2,
        scoring=SCORING_S2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0
    )

    # Fit on subset for speed
    search.fit(X_sub, y_sub)

    best_model = search.best_estimator_

    # ─── REFIT ON FULL DATA ──────────────────────────────────────────────────
    best_model.fit(X_train_s2, y_train_s2)

    s2_best_models[name]    = best_model
    s2_search_results[name] = search

    print(f'Best CV {SCORING_S2}: {search.best_score_:.4f}')

print('\nStage 2 optimization complete')

🔍 Fast search for 4 models (Stage 2)
   Scoring: f1_macro | CV: 3-fold | Halving factor: 3

  Logistic Regression...Best CV f1_macro: 0.1104
  Random Forest...Best CV f1_macro: 0.5709
  Extra Trees...Best CV f1_macro: 0.5561
  XGBoost...Best CV f1_macro: 0.3852

Stage 2 optimization complete


In [14]:
# ─── Оцінка всіх моделей Stage 2 ────────────────────────────────────────────
print('\n📊 STAGE 2 — Порівняння моделей на тестовій вибірці (оригінальний розподіл)')
s2_test_results = {}
s2_class_names  = list(le_s2.classes_)

for name, model in s2_best_models.items():
    res = evaluate_model(
        model, X_test_s2, y_test_s2,
        class_names=s2_class_names,
        stage_name=f'Stage 2 — {name}'
    )
    s2_test_results[name] = res

# Зведена таблиця
rows_s2 = []
for name, res in s2_test_results.items():
    row = {'Model': name, 'F1-macro': res['f1_macro'], 'F1-weighted': res['f1_weighted']}
    for cls in s2_class_names:
        row[f'F1-{cls}'] = res['f1_per_class'].get(cls, 0)
    rows_s2.append(row)

s2_summary = pd.DataFrame(rows_s2).sort_values('F1-macro', ascending=False)
print('\n📋 ЗВЕДЕНА ТАБЛИЦЯ Stage 2:')
print(s2_summary.to_string(index=False, float_format='{:.4f}'.format))

s2_best_name  = s2_summary.iloc[0]['Model']
s2_best_model = s2_best_models[s2_best_name]
print(f'\n🏆 Найкраща модель Stage 2: {s2_best_name} (F1-macro={s2_summary.iloc[0]["F1-macro"]:.4f})')


📊 STAGE 2 — Порівняння моделей на тестовій вибірці (оригінальний розподіл)

  Stage 2 — Logistic Regression
  Accuracy  : 0.6659
  F1-macro  : 0.1599  ← КЛЮЧОВА МЕТРИКА
  F1-weighted: 0.5324

  Per-class F1 (з confusion matrix):
    Click       : F1=0.799  Recall=1.000
    Comment     : F1=0.000  Recall=0.000
    Like        : F1=0.000  Recall=0.000
    Purchase    : F1=0.000  Recall=0.000
    Share       : F1=0.000  Recall=0.000

              precision    recall  f1-score   support

       Click       0.67      1.00      0.80      8099
     Comment       0.00      0.00      0.00       828
        Like       0.00      0.00      0.00      2429
    Purchase       0.00      0.00      0.00       410
       Share       0.00      0.00      0.00       396

    accuracy                           0.67     12162
   macro avg       0.13      0.20      0.16     12162
weighted avg       0.44      0.67      0.53     12162


  Stage 2 — Random Forest
  Accuracy  : 0.9183
  F1-macro  : 0.9157  ← КЛЮ

In [ ]:
# ─── Візуалізація Stage 2 ────────────────────────────────────────────────────
n_models = len(s2_test_results)
fig = plt.figure(figsize=(22, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)

ax_f1macro = fig.add_subplot(gs[0, 0])
compare_models_bar(s2_test_results, 'f1_macro', 'Stage 2: F1-macro', ax_f1macro)

ax_perclass = fig.add_subplot(gs[0, 1:])
model_names = list(s2_test_results.keys())
x = np.arange(len(s2_class_names))
width = 0.8 / len(model_names)
palette = sns.color_palette('tab10', len(model_names))
for i, (mname, res) in enumerate(s2_test_results.items()):
    vals = [res['f1_per_class'].get(c, 0) for c in s2_class_names]
    ax_perclass.bar(x + i*width - 0.4 + width/2, vals, width, label=mname, color=palette[i])
ax_perclass.set_xticks(x)
ax_perclass.set_xticklabels(s2_class_names, rotation=15)
ax_perclass.set_ylim(0, 1.0)
ax_perclass.set_title('Stage 2: Per-class F1 (збалансоване навчання)', fontweight='bold')
ax_perclass.set_ylabel('F1-score')
ax_perclass.legend(fontsize=8)

# Confusion matrix найкращої моделі Stage 2
ax_cm = fig.add_subplot(gs[1, :])
best_cm_s2 = s2_test_results[s2_best_name]['confusion_matrix']

# Normalize row-wise (percentages per true class)
cm_percent = best_cm_s2.astype('float') / best_cm_s2.sum(axis=1, keepdims=True) * 100

sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=s2_class_names, yticklabels=s2_class_names, ax=ax_cm,
            linewidths=0.5)

ax_cm.set_title(f'Confusion Matrix: {s2_best_name} (Stage 2) — % по рядках',
                fontweight='bold')
ax_cm.set_xlabel('Прогноз')
ax_cm.set_ylabel('Реальність')

plt.suptitle('Stage 2: Action Classifier — навчання на рівномірному розподілі',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Важливість ознак — всі три стадії

In [ ]:
def get_feature_importances(model, feature_names):
    """Витягує важливості ознак з будь-якої моделі (RF, ET, GB, Pipeline)."""
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names)
    elif hasattr(model, 'named_steps'):
        last = list(model.named_steps.values())[-1]
        if hasattr(last, 'feature_importances_'):
            return pd.Series(last.feature_importances_, index=feature_names)
        elif hasattr(last, 'coef_'):
            return pd.Series(np.abs(last.coef_).mean(axis=0), index=feature_names)
    return None


def plot_fi(model, feature_names, title, ax, top_n=15, color='steelblue'):
    fi = get_feature_importances(model, feature_names)
    if fi is None:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return
    fi_top = fi.sort_values(ascending=False).head(top_n)
    fi_pct = fi_top / fi_top.sum() * 100
    fi_pct.sort_values().plot(kind='barh', ax=ax, color=color)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Відносна важливість (%)')
    for bar, val in zip(ax.patches, fi_pct.sort_values()):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8)
    ax.set_xlim(0, fi_pct.max() * 1.25)


fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plot_fi(
    s1_best_model, feature_names,
    f'Stage 1: {s1_best_name}\n(Impression vs Engaged)',
    axes[0], color='#4878CF'
)

plot_fi(
    s2_best_model, feature_names,
    f'Stage 2: {s2_best_name}\n(Action Classifier)',
    axes[1], color='#6ACC65'
)

plt.suptitle(
    'Feature Importance — Two-Stage Pipeline (після оптимізації)',
    fontsize=16, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.show()

# Перетин топ-10 ознак (оновлено: без S3)
top10 = {}
for stage, model in [('S1', s1_best_model), ('S2', s2_best_model)]:
    fi = get_feature_importances(model, feature_names)
    if fi is not None:
        top10[stage] = set(fi.sort_values(ascending=False).head(10).index)

if len(top10) == 2:
    universal = top10['S1'] & top10['S2']
    print(f'\n🔗 Універсальні ознаки (топ-10 у обох стадіях):')
    for f in sorted(universal):
        print(f'  • {f}')

---
## 12. Підсумковий звіт — Фінальне порівняння всіх стадій

In [17]:
print('='*70)
print('ПІДСУМКОВИЙ ЗВІТ — ОПТИМІЗОВАНИЙ TWO-STAGE PIPELINE')
print('='*70)

print()
print('🔧 Застосовані виправлення:')
print('  #1 Scoring: f1_macro + per-class F1 з confusion matrix (не accuracy)')
print('  #2 Балансування: Impression = Engaged (50/50 downsampling)')
print(f'     Розмір Stage 1: {len(X_s1):,} рядків замість {len(X):,}')
print('  #3 Рівномірний розподіл активних дій: oversampling до 40% від max-класу')
print(f'     Stage 2 min-ratio: {MIN_RATIO:.0%} від Click')
print('  #4 Порівняння моделей: LR, Random Forest, Extra Trees, Gradient Boosting + XGBoost')
print(f'  #5 Automated tuning: RandomizedSearchCV ({N_ITER} ітерацій, {CV_FOLDS}-fold CV)')

print()
print('─'*70)
print('STAGE 1 — Binary Filter (Impression vs Engaged)')
print('─'*70)
for _, row in s1_summary.iterrows():
    prefix = '🏆' if row['Model'] == s1_best_name else '  '
    print(f"  {prefix} {row['Model']:25s}: F1-macro={row['F1-macro']:.4f}  "
          f"F1-Engaged={row['F1-Engaged']:.4f}")

print()
print('─'*70)
print('STAGE 2 — Action Classifier (Click/Like/Purchase/Share/Comment)')
print('─'*70)
for _, row in s2_summary.iterrows():
    prefix = '🏆' if row['Model'] == s2_best_name else '  '
    print(f"  {prefix} {row['Model']:25s}: F1-macro={row['F1-macro']:.4f}  "
          f"F1-weighted={row['F1-weighted']:.4f}")
print()
print('─'*70)
print('🎯 РЕКОМЕНДОВАНІ МОДЕЛІ ПАЙПЛАЙНУ:')
print(f'  Stage 1 Filter:    {s1_best_name}')
print(f'  Stage 2 Actions:   {s2_best_name}')
print('='*70)

ПІДСУМКОВИЙ ЗВІТ — ОПТИМІЗОВАНИЙ TWO-STAGE PIPELINE

🔧 Застосовані виправлення:
  #1 Scoring: f1_macro + per-class F1 з confusion matrix (не accuracy)
  #2 Балансування: Impression = Engaged (50/50 downsampling)
     Розмір Stage 1: 121,620 рядків замість 403,967
  #3 Рівномірний розподіл активних дій: oversampling до 40% від max-класу
     Stage 2 min-ratio: 40% від Click
  #4 Порівняння моделей: LR, Random Forest, Extra Trees, Gradient Boosting + XGBoost


NameError: name 'N_ITER' is not defined

In [ ]:
# ─── Фінальна зведена візуалізація ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (results, best, title) in zip(
    axes,
    [
        (s1_test_results, s1_best_name, 'Stage 1\nImpression vs Engaged'),
        (s2_test_results, s2_best_name, 'Stage 2\nAction Classifier'),
        (s3_test_results, s3_best_name, 'Stage 3\nClick vs Purchase'),
    ]
):
    names = list(results.keys())
    vals  = [results[n]['f1_macro'] for n in names]
    colors_bar = ['#2ecc71' if n == best else '#3498db' for n in names]
    bars = ax.bar(names, vals, color=colors_bar, edgecolor='white')
    ax.set_ylim(0, min(max(vals) * 1.3, 1.0))
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('F1-macro')
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Порівняння моделей по всіх стадіях пайплайну ( обрана модель)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()